# 06 - Probe vs counterfactuals (CPU, ~10 min)

**Settings:** Accelerator **None (CPU)** | Internet **On**. No secrets or dataset input needed (all data is public on HF, pinned revisions).

**Question:** the deployed probe flagged only 48 % of the counterfactual hop-1 failures in causal patching. Is that the probe or its strict threshold, and does any of Anish's retrained probes (`probe_suite_v1/tfidf`: tuned LR, SVM, MLP) do better?

`scripts/probe_cf_eval.py` scores, from the **stored** hidden states (no model), every probe on:
* the 505 TF-IDF counterfactuals (all hop-1 failures; main numbers = test split), by failure kind,
* their clean seeds (hop 1 correct, same context) -> false alarms and AUROC CF vs seed,
* the natural TF-IDF test hop-1 rows (hedge / confident failures, successes) as the reference,

at each probe's own operating threshold and at its default threshold (threshold effect).

Download `probe_cf_eval.zip` from the Output tab (or paste `results.md`).

In [ ]:
import os, subprocess, sys, zipfile
REPO_URL = 'https://github.com/ANLPproject/LinguaFranca.git'
WORK = '/kaggle/working/LinguaFranca'
if not os.path.exists(WORK):
    subprocess.run(['git', 'clone', '--depth', '1', '-b', 'cf-tfidf', REPO_URL, WORK], check=True)
else:
    subprocess.run(['git', '-C', WORK, 'pull'], check=True)
os.chdir(WORK)
print(subprocess.run(['git', 'log', '-1', '--oneline'], capture_output=True, text=True).stdout)
import sklearn; print('sklearn', sklearn.__version__, '(MLP probes were pickled with 1.6.1; skipped with a message if they cannot load)')

In [ ]:
rc = subprocess.run([sys.executable, 'scripts/probe_cf_eval.py', '--out', '/kaggle/working/probe_cf_eval']).returncode
assert rc == 0, 'failed - see the log above'
with zipfile.ZipFile('/kaggle/working/probe_cf_eval.zip', 'w') as z:
    for f in ('results.md', 'results.json'):
        z.write(f'/kaggle/working/probe_cf_eval/{f}', f)
print('saved /kaggle/working/probe_cf_eval.zip')